# Analzye collected data

Here, we gather the data collected from the different models and compare them to human data.

### Import

In [127]:
import pandas as pd
import plotly.express as px
import os
from pathlib import Path
import  matplotlib.pyplot as plt

### Functions

In [125]:
def read_scores_from_model(model):

    model_name = model + "_"
    files = list(Path("results").glob(f"{model_name}*.csv"))

    df = pd.DataFrame([])

    print(f"Gathered data for model {model} from the following files.")
    for f in files:
        print(f.name)
        df_data = pd.read_csv(f'results/{f.name}')
        df_data["run"] = f.name[-5:-4]
        df = pd.concat([df, df_data], ignore_index=True)

    df["model"] = model
    df["id"] = df["id"].astype(int)

    return df

def summarize_scores_from_model(df, model):
    summary = df.groupby("id").agg({"score": ["mean", "std", "sem"]}).reset_index()
    summary.columns = summary.columns.get_level_values(1)
    #summary.columns = ['id', f'{model}_mean', f'{model}_std', f'{model}_sem']
    summary.columns = ['id', 'score_mean', 'score_std', 'score_sem']
    print(f"Summarized data for model {model}.")

    return summary


def collate_human_and_llm_data_wide(MODELS: list) -> pd.DataFrame:
    
    # fetch human data
    df = pd.read_csv("data/iCO-Eval2_summarizedRatings.csv", sep=";")

    # fetch, aggregate and merge LLm data
    for model in MODELS:
        df_model = read_scores_from_model(model)
        summary_model = summarize_scores_from_model(df_model, model)
        summary_model.columns = ['id', f'{model}_mean', f'{model}_std', f'{model}_sem']
        print(summary_model.head())
        df = pd.merge(df, summary_model, left_on="CODE", right_on="id", how="left").drop(columns='id')
        

    return df


def collate_human_and_llm_data_long(MODELS: list) -> pd.DataFrame:
    
    # fetch human data
    df = pd.read_csv("data/iCO-Eval2_summarizedRatings.csv", sep=";")
    df.drop(columns=["SET (1=SA-matched;2=non-SA-matched)", "CONDITION", "CONTEXT QUESTION", "CRITICAL UTTERANCE", "CORRECT RESPONSE (0=no; 1=yes)", "FUN_std", "FUN_sem", "COH_std", "COH_sem", "DIR_std", "DIR_sem", "PRE_std", "PRE_sem", "SSI_std", "SSI_sem", "CER_std", "CER_sem"], inplace=True) #this is not really necessary

    # put them on the same scale as the LLm scores (0-1)
    df["FUN_mean"] = rescale(df["FUN_mean"], 1, 7)
    df["COH_mean"] = rescale(df["COH_mean"], 1, 7)
    df["DIR_mean"] = rescale(df["DIR_mean"], 1, 7)
    df["PRE_mean"] = rescale(df["PRE_mean"], 1, 7)
    df["SSI_mean"] = rescale(df["SSI_mean"], 1, 7)
    df["CER_mean"] = rescale(df["CER_mean"], 1, 4)
    df["evaluator"] = "human"

    # fetch, aggregate and merge LLm data
    for model in MODELS:
        df_model = read_scores_from_model(model)
        summary_model = summarize_scores_from_model(df_model, model)
        summary_model  = summary_model[["id", "score_mean"]]
        summary_model.rename(columns={"score_mean": "FUN_mean", "id": "CODE"}, inplace=True)
        summary_model["evaluator"] = model
        df = pd.concat([df, summary_model], axis=0, join='outer').reset_index(drop=True)

    return df

def rescale(series: pd.Series, min: int, max: int) -> pd.Series:
    return (series - min) / (max - min)

### Aggregation

In [126]:
MODELS = [
    "gpt-5.4",
    "gpt-5.4-nano-2026-03-17",
    "gpt-5.4-mini-2026-03-17",
]

df_wide =  collate_human_and_llm_data_wide(MODELS)
df_long = collate_human_and_llm_data_long(MODELS)


Gathered data for model gpt-5.4 from the following files.
gpt-5.4_score_run1.csv
gpt-5.4_score_run2.csv
gpt-5.4_score_run3.csv
Summarized data for model gpt-5.4.
     id  gpt-5.4_mean  gpt-5.4_std  gpt-5.4_sem
0  1101      0.050000     0.000000     0.000000
1  1102      0.980000     0.000000     0.000000
2  1103      0.850000     0.000000     0.000000
3  1104      0.970000     0.000000     0.000000
4  1105      0.716667     0.028868     0.016667
Gathered data for model gpt-5.4-nano-2026-03-17 from the following files.
gpt-5.4-nano-2026-03-17_score_run2.csv
gpt-5.4-nano-2026-03-17_score_run3.csv
gpt-5.4-nano-2026-03-17_score_run1.csv
Summarized data for model gpt-5.4-nano-2026-03-17.
     id  gpt-5.4-nano-2026-03-17_mean  gpt-5.4-nano-2026-03-17_std  \
0  1101                      0.950000                     0.000000   
1  1102                      0.916667                     0.028868   
2  1103                      0.933333                     0.028868   
3  1104                     

### Visualization

In [133]:
fig = px.strip(df_long, y="FUN_mean", x="evaluator", color="evaluator", stripmode='group')
fig.update_traces(marker_size=3)
fig.show()